# 01 — Bronze Ingestion

**Purpose:** Land NYC TLC Yellow Taxi trip data (Jan–Jun 2024) and the taxi-zone lookup into the Bronze Lakehouse as immutable Delta tables.

**Source:** NYC TLC public CloudFront CDN (`https://d37ci6vzurychx.cloudfront.net/trip-data/`)

**Output:**
- `bronze.yellow_tripdata` — Delta table, partitioned by `year`/`month`
- `bronze.taxi_zone_lookup` — Delta table, reference data

**Design notes:**
- Bronze is **append-only and untransformed**. Schema is preserved exactly as the source publishes it. Cleanup, type changes, and dedup happen in Silver, not here.
- Audit columns (`_ingested_at`, `_source_file`) are added on write so we can trace any row in the lake back to its source file and ingestion run.
- The download step is idempotent — already-downloaded files are skipped, so the cell can be re-run safely.

**Attach this notebook to the `bronze` Lakehouse as the default Lakehouse before running.**

In [ ]:
# --- Imports ---
import requests
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# --- Config ---
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

MONTHS = [
    "2024-01", "2024-02", "2024-03",
    "2024-04", "2024-05", "2024-06",
]

# Fabric mounts the default Lakehouse at /lakehouse/default
# Files/ is the unmanaged file area; Tables/ is the managed Delta table area
LANDING_PATH = "/lakehouse/default/Files/raw"
Path(LANDING_PATH).mkdir(parents=True, exist_ok=True)

print(f"Landing zone: {LANDING_PATH}")
print(f"Months to ingest: {len(MONTHS)} ({MONTHS[0]} → {MONTHS[-1]})")

## Step 1 — Download to landing zone

Pull the monthly Parquet files and the zone-lookup CSV into `Files/raw/`. This is the immutable landing area — once a file is here it never changes, and Silver/Gold transformations always read from Delta tables, never from these raw files.

Why land first instead of reading from HTTP directly into Spark?

1. **Reliability.** Spark over HTTPS depends on the runtime's HTTP client; local landing is bullet-proof.
2. **Reproducibility.** If the source site goes down or republishes a corrected file, we still have the version we ingested.
3. **Auditability.** `_source_file` in the Delta table points back to a real file in the lake.

In [ ]:
def download_if_missing(url: str, dest: str) -> dict:
    """Download a URL to dest if not already present. Returns metadata dict."""
    dest_path = Path(dest)
    if dest_path.exists():
        size_mb = dest_path.stat().st_size / (1024 * 1024)
        return {"file": dest_path.name, "status": "skipped", "size_mb": round(size_mb, 1)}

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with open(dest, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
            f.write(chunk)

    size_mb = dest_path.stat().st_size / (1024 * 1024)
    return {"file": dest_path.name, "status": "downloaded", "size_mb": round(size_mb, 1)}


# Download monthly trip files
results = []
for month in MONTHS:
    filename = f"yellow_tripdata_{month}.parquet"
    url = f"{BASE_URL}/{filename}"
    dest = f"{LANDING_PATH}/{filename}"
    result = download_if_missing(url, dest)
    results.append(result)
    print(f"  [{result['status']:>10}] {result['file']:<40} {result['size_mb']:>7.1f} MB")

# Download zone lookup
zone_dest = f"{LANDING_PATH}/taxi_zone_lookup.csv"
zone_result = download_if_missing(ZONE_LOOKUP_URL, zone_dest)
print(f"  [{zone_result['status']:>10}] {zone_result['file']:<40} {zone_result['size_mb']:>7.2f} MB")

total_mb = sum(r["size_mb"] for r in results) + zone_result["size_mb"]
print(f"\n  Total landed: {total_mb:.1f} MB across {len(results) + 1} files")

## Step 2 — Read raw Parquet into Spark

Wildcard read across all 6 monthly files. Spark infers the schema from the Parquet metadata.

A note on TLC schema drift: the TLC has changed the trip schema several times across years. For our 2024 slice the schema is stable across all 6 months, but if you extend this notebook to older data you'll want to either enforce a schema explicitly or use `mergeSchema=True`.

In [ ]:
# Read all monthly files in one go
raw_df = spark.read.parquet(f"{LANDING_PATH}/yellow_tripdata_*.parquet")

print(f"Row count: {raw_df.count():,}")
print(f"Column count: {len(raw_df.columns)}")
print()
print("Schema:")
raw_df.printSchema()

In [ ]:
# Quick eyeball of the data — first 5 rows of key columns
raw_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "tip_amount",
    "total_amount"
).show(5, truncate=False)

## Step 3 — Add audit columns and write to Bronze Delta

Three audit columns let any downstream consumer trace a row back to its origin:

| Column | Purpose |
|--------|---------|
| `_ingested_at` | UTC timestamp this Bronze write happened — defines an ingestion run |
| `_source_file` | The Parquet filename the row came from — defines provenance |
| `_pickup_year` / `_pickup_month` | Derived partition keys |

We partition the Delta table by `_pickup_year` / `_pickup_month` because that matches how the data is naturally queried (time slicing) and how it was published (monthly files). Partition pruning will keep Silver-stage transformations fast.

In [ ]:
ingestion_ts = datetime.now(timezone.utc)

bronze_df = (
    raw_df
    # input_file_name() returns the full path; basename it for readability
    .withColumn("_source_file", F.regexp_extract(F.input_file_name(), r"([^/]+)$", 1))
    .withColumn("_ingested_at", F.lit(ingestion_ts).cast(TimestampType()))
    .withColumn("_pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("_pickup_month", F.month("tpep_pickup_datetime"))
)

# Write to a managed Delta table in the default Lakehouse.
# overwriteSchema=True is a safety net for the first run; in production you'd
# typically use append + schema evolution policies instead.
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_pickup_year", "_pickup_month")
    .saveAsTable("yellow_tripdata")
)

print(f"✓ Wrote bronze.yellow_tripdata at {ingestion_ts.isoformat()}")

In [ ]:
# Zone lookup — small reference table, no partitioning needed
zone_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{LANDING_PATH}/taxi_zone_lookup.csv")
    .withColumn("_source_file", F.lit("taxi_zone_lookup.csv"))
    .withColumn("_ingested_at", F.lit(ingestion_ts).cast(TimestampType()))
)

(
    zone_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("taxi_zone_lookup")
)

print(f"✓ Wrote bronze.taxi_zone_lookup ({zone_df.count()} zones)")

## Step 4 — Validation

Three checks. If any of these surprise us, we don't move on to Silver until we understand why.

In [ ]:
# Check 1: row counts per partition match expectations
# (Each month should have roughly 2.5-3.5M rows of yellow taxi trips)
print("--- Check 1: row counts per partition ---")
spark.sql("""
    SELECT _pickup_year, _pickup_month, COUNT(*) AS row_count
    FROM yellow_tripdata
    GROUP BY _pickup_year, _pickup_month
    ORDER BY _pickup_year, _pickup_month
""").show()

In [ ]:
# Check 2: data ranges look right
# Bronze IS noisy — we'll see trips outside our intended window because
# TLC publication occasionally includes stragglers from adjacent months.
# That's fine — Silver will handle it. We're just confirming nothing is wildly broken.
print("--- Check 2: pickup datetime range ---")
spark.sql("""
    SELECT
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        COUNT(DISTINCT _source_file) AS source_files
    FROM yellow_tripdata
""").show(truncate=False)

In [ ]:
# Check 3: audit columns populated
print("--- Check 3: audit column coverage ---")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(_ingested_at) AS rows_with_ingested_at,
        COUNT(_source_file) AS rows_with_source_file,
        COUNT(DISTINCT _source_file) AS distinct_source_files
    FROM yellow_tripdata
""").show()

## Done — Bronze ready for Silver

**What's in the lake now:**
- `bronze.yellow_tripdata` — partitioned by year/month, ~18M rows expected across Jan–Jun 2024
- `bronze.taxi_zone_lookup` — 265 zones

**What to do next:** Open `02_silver_transform.ipynb`. Silver will:
1. Apply schema enforcement and type casting
2. Dedupe (rare but does happen in TLC data)
3. Quarantine trips with impossible durations or distances
4. Standardize categoricals against the zone lookup and rate-code dictionary